# 诗歌生成

# 数据处理

In [21]:
import numpy as np
import tensorflow as tf
import collections
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import layers, optimizers, datasets

start_token = 'bos'
end_token = 'eos'

def process_dataset(fileName):
    examples = []
    with open(fileName, 'r', encoding='utf-8') as fd:
        for line in fd:
            line = line.strip()
            if not line:
                continue
            outs = line.split(':')
            if len(outs) >= 2:
                content = ''.join(outs[1:])
            else:
                content = line
            ins = [start_token] + list(content) + [end_token]
            if len(ins) > 200:
                continue
            examples.append(ins)
            
    counter = collections.Counter()
    for e in examples:
        for w in e:
            counter[w]+=1
    
    sorted_counter = sorted(counter.items(), key=lambda x: -x[1])  # 排序
    words, _ = zip(*sorted_counter)
    words = ('PAD', 'UNK') + words[:len(words)]
    word2id = dict(zip(words, range(len(words))))
    id2word = {word2id[k]:k for k in word2id}
    
    indexed_examples = [[word2id[w] for w in poem]
                        for poem in examples]
    seqlen = [len(e) for e in indexed_examples]
    
    instances = list(zip(indexed_examples, seqlen))
    
    return instances, word2id, id2word

def poem_dataset():
    instances, word2id, id2word = process_dataset('tangshi.txt')
    ds = tf.data.Dataset.from_generator(lambda: [ins for ins in instances], 
                                            (tf.int64, tf.int64), 
                                            (tf.TensorShape([None]),tf.TensorShape([])))
    ds = ds.shuffle(buffer_size=10240)
    ds = ds.padded_batch(100, padded_shapes=(tf.TensorShape([None]),tf.TensorShape([])))
    ds = ds.map(lambda x, seqlen: (x[:, :-1], x[:, 1:], seqlen-1))
    return ds, word2id, id2word

# 模型代码， 完成建模代码

In [22]:
class myRNNModel(keras.Model):
    def __init__(self, w2id):
        super(myRNNModel, self).__init__()
        self.v_sz = len(w2id)
        self.embed_layer = tf.keras.layers.Embedding(self.v_sz, 128)
        
        self.rnncell = tf.keras.layers.GRUCell(256)
        self.rnn_layer = tf.keras.layers.RNN(self.rnncell, return_sequences=True)
        self.dense = tf.keras.layers.Dense(self.v_sz)
        
    @tf.function(reduce_retracing=True)
    def call(self, inp_ids):
        '''
        此处完成建模过程，可以参考Learn2Carry
        '''
        inp_emb = self.embed_layer(inp_ids)
        h = self.rnn_layer(inp_emb)
        logits = self.dense(h)
        return logits
    
    @tf.function(reduce_retracing=True)
    def get_next_token(self, x, state):
        '''
        shape(x) = [b_sz,] 
        '''
    
        inp_emb = self.embed_layer(x) #shape(b_sz, emb_sz)
        h, state = self.rnncell.call(inp_emb, state) # shape(b_sz, h_sz)
        logits = self.dense(h) # shape(b_sz, v_sz)
        return logits, state

## 一个计算sequence loss的辅助函数，只需了解用途。

In [23]:
def mkMask(input_tensor, maxLen):
    shape_of_input = tf.shape(input_tensor)
    shape_of_output = tf.concat(axis=0, values=[shape_of_input, [maxLen]])

    oneDtensor = tf.reshape(input_tensor, shape=(-1,))
    flat_mask = tf.sequence_mask(oneDtensor, maxlen=maxLen)
    return tf.reshape(flat_mask, shape_of_output)


def reduce_avg(reduce_target, lengths, dim):
    """
    Args:
        reduce_target : shape(d_0, d_1,..,d_dim, .., d_k)
        lengths : shape(d0, .., d_(dim-1))
        dim : which dimension to average, should be a python number
    """
    shape_of_lengths = lengths.get_shape()
    shape_of_target = reduce_target.get_shape()
    if len(shape_of_lengths) != dim:
        raise ValueError(('Second input tensor should be rank %d, ' +
                         'while it got rank %d') % (dim, len(shape_of_lengths)))
    if len(shape_of_target) < dim+1 :
        raise ValueError(('First input tensor should be at least rank %d, ' +
                         'while it got rank %d') % (dim+1, len(shape_of_target)))

    rank_diff = len(shape_of_target) - len(shape_of_lengths) - 1
    mxlen = tf.shape(reduce_target)[dim]
    mask = mkMask(lengths, mxlen)
    if rank_diff!=0:
        len_shape = tf.concat(axis=0, values=[tf.shape(lengths), [1]*rank_diff])
        mask_shape = tf.concat(axis=0, values=[tf.shape(mask), [1]*rank_diff])
    else:
        len_shape = tf.shape(lengths)
        mask_shape = tf.shape(mask)
    lengths_reshape = tf.reshape(lengths, shape=len_shape)
    mask = tf.reshape(mask, shape=mask_shape)

    mask_target = reduce_target * tf.cast(mask, dtype=reduce_target.dtype)

    red_sum = tf.reduce_sum(mask_target, axis=[dim], keepdims=False)
    red_avg = red_sum / (tf.cast(lengths_reshape, dtype=tf.float32) + 1e-30)
    return red_avg

# 定义loss函数，定义训练函数

In [24]:
@tf.function(reduce_retracing=True)
def compute_loss(logits, labels, seqlen):
    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels)
    losses = reduce_avg(losses, seqlen, dim=1)
    return tf.reduce_mean(losses)

@tf.function(reduce_retracing=True)
def train_one_step(model, optimizer, x, y, seqlen):
    '''
    完成一步优化过程，可以参考之前做过的模型
    '''
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y, seqlen)
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

def train(epoch, model, optimizer, ds):
    loss = 0.0
    accuracy = 0.0
    for step, (x, y, seqlen) in enumerate(ds):
        loss = train_one_step(model, optimizer, x, y, seqlen)

        if step % 500 == 0:
            print('epoch', epoch, ': loss', loss.numpy())

    return loss

# 训练优化过程

In [25]:
optimizer = optimizers.Adam(0.001)
train_ds, word2id, id2word = poem_dataset()
model = myRNNModel(word2id)
_ = model(tf.constant([[word2id['bos']]], dtype=tf.int64))

for epoch in range(80):
    loss = train(epoch, model, optimizer, train_ds)

epoch 0 : loss 7.828409
epoch 1 : loss 7.7658653
epoch 2 : loss 6.713993
epoch 3 : loss 6.552783
epoch 4 : loss 6.4685607
epoch 5 : loss 6.374414
epoch 6 : loss 6.4182396
epoch 7 : loss 6.373991
epoch 8 : loss 6.291269
epoch 9 : loss 6.247258
epoch 10 : loss 6.274861
epoch 11 : loss 6.2642117
epoch 12 : loss 6.2116075
epoch 13 : loss 6.13801
epoch 14 : loss 6.0514917
epoch 15 : loss 6.108641
epoch 16 : loss 6.044467
epoch 17 : loss 6.012744
epoch 18 : loss 6.002803
epoch 19 : loss 5.9979644
epoch 20 : loss 5.918245
epoch 21 : loss 5.855856
epoch 22 : loss 5.8510714
epoch 23 : loss 5.804259
epoch 24 : loss 5.7597694
epoch 25 : loss 5.730393
epoch 26 : loss 5.710006
epoch 27 : loss 5.693126
epoch 28 : loss 5.6226735
epoch 29 : loss 5.5375
epoch 30 : loss 5.546936
epoch 31 : loss 5.507437
epoch 32 : loss 5.4452486
epoch 33 : loss 5.441721
epoch 34 : loss 5.4179735
epoch 35 : loss 5.372295
epoch 36 : loss 5.293583
epoch 37 : loss 5.3183055
epoch 38 : loss 5.262525
epoch 39 : loss 5.2673216

# 生成过程

In [27]:
def gen_sentence(begin_word, max_len=50, temperature=0.7, top_k=8):
    state = [tf.zeros(shape=(1, 256), dtype=tf.float32)]
    collect = [begin_word]
    cur_token = tf.constant([word2id.get(begin_word, word2id['UNK'])], dtype=tf.int32)
    banned_words = ['bos', 'eos', 'PAD', 'UNK']
    banned_ids = [word2id[w] for w in banned_words if w in word2id]
    for _ in range(max_len):
        inp_emb = model.embed_layer(cur_token)
        h, state = model.rnncell.call(inp_emb, state)
        logits = model.dense(h).numpy()[0] / temperature
        logits[banned_ids] = -1e9
        top_indices = np.argsort(logits)[-top_k:]
        top_logits = logits[top_indices]
        probs = np.exp(top_logits - np.max(top_logits))
        probs = probs / np.sum(probs)
        token_id = int(np.random.choice(top_indices, p=probs))
        word = id2word[token_id]
        if word in banned_words:
            break
        collect.append(word)
        cur_token = tf.constant([token_id], dtype=tf.int32)
    return ''.join(collect)

for begin_word in ['日', '红', '山', '夜', '湖', '海', '月']:
    print(begin_word, gen_sentence(begin_word))

日 日军青知不知。天子天子不见，小然赤能争。春天久不见老，君人不可褐。君君不见老，铁人无君天，吾知有心草。
红 红税。蛮之毂攒繁。脑涂原育，纵狄纵草。脑涂原骨，驰霍岧中。豺涂原立，铁华驰骤。著豺炮凤相突，小然松驰骤
山 山第万门不可邻，清来何时干。又来一骨无时，天骨无肯飞。总嘶君无为深，铁风无肯珍。盖人天子不肯涉，小管凤
夜 夜遥莱围帽织帽织织织莱莱效织织枿琥织槔败效缧织莱枿莱织莱织莱莱织败效槔珀莱织枿织织莱莱莱织织效织效虣枿
湖 湖何不见月，高江何风事。主人不见开华间，惜哉万草高台下。又来一深久无君。又如天骨下何逡巡，吾人不见主父
海 海上春人。主人一见赤木死，小管凤吟门上。天骨络深龙然深。盖人有为何逡巡，何风有下风。又在天骨无时发，小
月 月税，沈兵子不见，清流一服。天里一传神，蹭跣不能鳞。著君天如相骨，小管争岧人。又有相思君，天里不肯驰骤
